In [0]:
# ============================================================
# Bronze Ingestion: Warehouse Inventory
# Notebook: bronze/ingest_warehouse_inventory
# Trigger: Databricks Workflow — every hour
# ============================================================

from pyspark.sql.functions import (
    current_timestamp, input_file_name, lit
)
from pyspark.sql.types import (
    StructType, StructField, StringType,
    IntegerType, LongType, TimestampType
)

In [0]:
SOURCE_PATH       = "/Volumes/mini_project_team_a3t7/default/mini_project/raw/warehouse_inventory/"
CHECKPOINT_PATH   = "/Volumes/mini_project_team_a3t7/default/mini_project/checkpoint/bronze/warehouse_inventory/"
SCHEMA_PATH       = "/Volumes/mini_project_team_a3t7/default/mini_project/checkpoint/bronze/warehouse_inventory_schema/"
DESTINATION_PATH  = "/Volumes/mini_project_team_a3t7/default/mini_project/bronze/warehouse_inventory/"

# For Parquet we define the schema explicitly.
# Never use inferSchema on Parquet in production —
# it reads all files to infer, defeating the purpose of Auto Loader.
warehouse_schema = StructType([
    StructField("warehouse_id",    StringType(),  nullable=False),
    StructField("product_id",      StringType(),  nullable=False),
    StructField("current_stock",   LongType(), nullable=True),
    StructField("reserved_stock",  LongType(), nullable=True),
    StructField("available_stock", LongType(), nullable=True),
    StructField("reorder_level",   LongType(), nullable=True),
    StructField("max_stock",       LongType(), nullable=True),
    StructField("last_updated",    StringType(), nullable=True),
    StructField("location_zone",   StringType(),  nullable=True),
])

In [0]:
raw_df = (
    spark.readStream
         .format("cloudFiles")                          # Auto Loader
         .option("cloudFiles.format", "parquet")        # source format
         .option("cloudFiles.schemaLocation", SCHEMA_PATH)
         .schema(warehouse_schema)                      # explicit schema
         .load(SOURCE_PATH)
)

enriched_df = (
    raw_df
    .withColumn("_ingested_at",  current_timestamp())   # when we processed it
    .withColumn("_pipeline_run", lit("bronze_warehouse_inventory"))
)

In [0]:
# Add Watermark
watermarked_df = (
    enriched_df
    .withWatermark("_ingested_at", "10 minutes")
)

In [0]:
# WRITE STREAM

(
    watermarked_df
    .writeStream
    .format("delta")
    .outputMode("append")                               # Bronze is always append
    .option("checkpointLocation", CHECKPOINT_PATH)
    .option("mergeSchema", "true")                      # safe schema evolution
    .trigger(availableNow=True)                         # process all new files, then stop
    .start(DESTINATION_PATH)
)

In [0]:
# spark.read.format('delta').load(DESTINATION_PATH).display()